In [1]:
%pip install rank_bm25 sentence-transformers numpy scikit-learn

In [2]:
documents = [
    "Neural networks are used in deep learning.",
    "Cats are small animals.",
    "Transformers revolutionized NLP.",
    "Dogs are loyal animals."
]

query = "How do attention models work?"

# Naive keyword matching
results = [doc for doc in documents if any(word in doc.lower() for word in query.lower().split())]

print("Naive Results:", results)

Naive Results: ['Dogs are loyal animals.']


In [3]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

tokenized_query = query.lower().split()
scores = bm25.get_scores(tokenized_query)

for doc, score in zip(documents, scores):
    print(doc, ":", score)

Neural networks are used in deep learning. : 0.0
Cats are small animals. : 0.0
Transformers revolutionized NLP. : 0.0
Dogs are loyal animals. : 0.0


In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

doc_embeddings = model.encode(documents)
query_embedding = model.encode([query])

scores = cosine_similarity(query_embedding, doc_embeddings)[0]

for doc, score in zip(documents, scores):
    print(doc, ":", score)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Neural networks are used in deep learning. : 0.32306007
Cats are small animals. : 0.011411633
Transformers revolutionized NLP. : 0.18666357
Dogs are loyal animals. : 0.076735765


In [5]:
import numpy as np

def colbert_score(query_emb, doc_emb):
    return np.max(np.dot(query_emb, doc_emb.T))

# simulate token embeddings
q = np.random.rand(5, 10)
d = np.random.rand(7, 10)

print("ColBERT Score:", colbert_score(q, d))

ColBERT Score: 3.863773364228867


In [6]:
def rrf(rank_lists, k=60):
    scores = {}
    for rank_list in rank_lists:
        for rank, doc_id in enumerate(rank_list):
            scores[doc_id] = scores.get(doc_id, 0) + 1/(k+rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

bm25_ranks = [0,1,2,3]
sbert_ranks = [2,0,1,3]

print(rrf([bm25_ranks, sbert_ranks]))

[(0, 0.03306010928961749), (2, 0.03279569892473118), (1, 0.03252247488101534), (3, 0.031746031746031744)]
